# Example 7.8.2 / 1: Request a Pool and Load Each Solution

Companion notebook for **7.8.2 Alternative Solutions and Solution Pools** in the AMPL lab Documentation, section "Request a Pool and Load Each Solution". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Request a Pool and Load Each Solution

MP-based drivers that support pools share the option `sol:stub`: with `sol:stub=pool`, the
driver writes the solutions it keeps to the files `pool1.sol`, `pool2.sol`, ... and returns their
number in the suffix `Initial.nsol` (`Initial` is AMPL's name for the default problem). The AMPL
command `solution pool2.sol;` loads one of them into the variables. amplpy has no method for
`solution`, so the Python version runs it with `ampl.eval`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;  # weight capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;  # volume capacity

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="gcg", gcg_options="sol:stub=pool", verbose=False)
count = int(ampl.get_value("Initial.nsol"))
print(ampl.solve_result, "| solutions in the pool:", count)
for k in range(1, count + 1):
    ampl.eval(f"solution pool{k}.sol;")
    taken = [i for i, v in ampl.var["Take"].to_dict().items() if v > 0.5]
    print(k, ampl.obj["Total_Value"].value(), taken)

solved | solutions in the pool: 5
1 153.0 ['i1', 'i2', 'i5', 'i6']
2 145.0 ['i2', 'i6', 'i8']
3 142.0 ['i3', 'i6', 'i8']
4 138.0 ['i6', 'i7', 'i8']
5 0.0 []


### Pure AMPL: data.dat and model.run

`ampl.get_value("Initial.nsol")` reads the count. After each `solution` command the variables
hold that stored solution, so `to_dict()` and the objective show it. Solution 1 is the optimum,
153. The others are the improving solutions GCG met during its search, down to the empty
knapsack it started from: the pool is a by-product of the search, not a list of the next-best
plans. Two plans worth 141 are missing from it.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gcg;
option solver_msg 0;
option gcg_options 'sol:stub=pool';
solve;
printf "%s | solutions in the pool: %d\n", solve_result, Initial.nsol;
for {k in 1..Initial.nsol} {
    solution ("pool" & k & ".sol");
    printf "%d %g", k, Total_Value;
    printf {i in ITEMS: Take[i] > 0.5} " %s", i;
    printf "\n";
}

GCG 4.0.2:   sol:stub = pool

suffix nsol OUT;
suffix npool OUT;
solved | solutions in the pool: 5
1 153 i1 i2 i5 i6
2 145 i2 i6 i8
3 142 i3 i6 i8
4 138 i6 i7 i8
5 0


### Another way in amplpy

To ask for the best few solutions instead, Gurobi, CPLEX and Xpress also accept `sol:poollimit`, the most solutions to keep, and Gurobi and
CPLEX accept `sol:poolmode`; in Gurobi `sol:poolmode=2` searches systematically for the best
`sol:poollimit` solutions. With the same `model.mod` and `data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [11]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="gurobi", gurobi_options="sol:stub=pool sol:poollimit=5 sol:poolmode=2",
           verbose=False)
for k in range(1, int(ampl.get_value("Initial.nsol")) + 1):
    ampl.eval(f"solution pool{k}.sol;")
    taken = [i for i, v in ampl.var["Take"].to_dict().items() if v > 0.5]
    print(k, ampl.obj["Total_Value"].value(), taken)

1 153.0 ['i1', 'i2', 'i5', 'i6']
2 145.0 ['i2', 'i6', 'i8']
3 142.0 ['i3', 'i6', 'i8']
4 141.0 ['i2', 'i3', 'i8']
5 141.0 ['i1', 'i4', 'i6', 'i8']
